# CS383: Data Science and Machine Learning
## Lecture 7 Exercises — Linear Regression

Fill in every `__________` blank, then run all cells top to bottom. When you've completed this
notebook, download it (File → Save and Export Notebook As → Notebook (.ipynb), or the **Download**
button in the toolbar) and submit it on BrightSpace under **Lecture 7 Exercise** as a Jupyter Notebook
(.ipynb) file.

### Setup — NYC restaurant inspections

Run this first — it rebuilds the same row-level restaurant inspections dataset from the lecture
(one row per violation citation), plus the rare-cuisine grouping from Lecture 6.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score"]).reset_index(drop=True)
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 4000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese",
                       "Caribbean", "Bakery", "Coffee/Tea", "Chicken", "Thai", "Vietnamese"]

    is_critical = rng.integers(0, 2, size=n)
    score = rng.integers(0, 71, size=n)

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n),
        "score": score,
        "critical_flag": np.where(is_critical == 1, "Critical", "Not Critical"),
        "is_critical": is_critical,
    })
    live = False

cuisine_counts = inspections_df["cuisine_description"].value_counts()
common_cuisines = cuisine_counts[cuisine_counts >= 100].index
inspections_df["cuisine_grouped"] = inspections_df["cuisine_description"].where(
    inspections_df["cuisine_description"].isin(common_cuisines), other="Other"
)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} violation records")
inspections_df[["boro", "cuisine_grouped", "is_critical", "score"]].head()

---

## Exercise 1 — Put It Together

**Scenario:** predict a restaurant's inspection `score` using both its categorical features (`boro`,
`cuisine_grouped`) and its numeric feature (`is_critical`) together — combining Lecture 6's
encoding/scaling with today's regression, the same kind of full workflow your capstone will need.
Nobody is promising this will predict `score` well — that's exactly what you're about to find out,
honestly.

### Step 1 — Split first

In [ ]:
numeric_features = ["is_critical"]
categorical_features = ["boro", "cuisine_grouped"]

X = inspections_df[numeric_features + categorical_features]
y = inspections_df["score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=__________, random_state=383)

print("Training rows:", len(X_train))
print("Test rows:    ", len(X_test))

### Step 2 — Build a `ColumnTransformer` for both feature types

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", __________(), numeric_features),
    ("cat", __________(handle_unknown="ignore"), categorical_features),
])

X_train_ready = preprocessor.__________(X_train)
X_test_ready = preprocessor.__________(X_test)

print("Training features shape:", X_train_ready.shape)
print("Test features shape:    ", X_test_ready.shape)

### Step 3 — Fit a linear regression model

In [ ]:
model = __________()
model.fit(X_train_ready, y_train)

y_pred = model.__________(X_test_ready)

### Step 4 — Evaluate with MAE, RMSE, and R²

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(__________(y_test, y_pred))
r2 = __________(y_test, y_pred)

print(f"MAE:  {mae:.2f} points")
print(f"RMSE: {rmse:.2f} points")
print(f"R²:   {r2:.4f}")

### Step 5 — Plot the residuals

In [ ]:
residuals = y_test - __________

plt.scatter(y_pred, residuals, alpha=0.4, s=15)
plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("Predicted score")
plt.ylabel("Residual")
plt.title("Residuals vs. Predicted Score")
plt.show()

### Step 6 — Explain it back

In 2-3 sentences: what does your R² from Step 4, together with the residual plot from Step 5, tell you
about this model? Be specific about what the residual plot does and doesn't tell you on its own (the
lecture's Part 4 covered this distinction directly).

**Your explanation:**

---

## Exercise 2 — Reflection (Exit Ticket)

Answer the following in your own words.

1. Why do MAE and RMSE give different numbers for the same model? What does a big gap between them tell you?
2. Your R² in Step 4 was probably very close to zero, maybe even negative. Does that automatically mean
   there's a bug in your code? How would you actually check the difference between "correct code, weak
   signal" and "a real mistake"?
3. Suppose your residual plot from Step 5 had looked perfectly flat and patternless. Would that have
   proven the model was good? Why or why not?
4. In your own words, why do we fit the `ColumnTransformer` (and the model) on `X_train` only, not the
   full dataset?
5. What question do you still have about regression heading into Week 9 (Evaluation Metrics)?

**Your responses:**

1.
2.
3.
4.
5. 

## Optional Challenge

Apply the same workflow to NYC 311's `resolution_time_hours`, and try the log-transform trick from the
lecture yourself.

### Setup — NYC 311

In [ ]:
import os

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_idx = rng.integers(0, n_days, size=n)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")
    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(complaint_types, size=n),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour

# Same cleanup as the lecture notebook: keep only complaints that actually closed, drop the tiny
# "Unspecified" borough category, and group rare complaint types (100+ distinct values in the real
# data, many with just a handful of rows) into "Other" so a one-hot encoding can't treat a single
# rare row as a reliable pattern.
complaints_df = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)
standard_boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
complaints_df = complaints_df[complaints_df["borough"].isin(standard_boroughs)].reset_index(drop=True)
top_types = complaints_df["complaint_type"].value_counts().nlargest(15).index
complaints_df["complaint_grouped"] = complaints_df["complaint_type"].where(
    complaints_df["complaint_type"].isin(top_types), "Other"
)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df[["complaint_type", "complaint_grouped", "borough", "hour_filed", "resolution_time_hours"]].head()

### Step 1 — Split, encode, and fit

Use `complaint_grouped`, `borough`, and `hour_filed` as features and `resolution_time_hours` as the
target. Split, build a `ColumnTransformer` (one-hot for the two categoricals, scale `hour_filed`), then
fit a `LinearRegression`.

In [ ]:
# Your code here


### Step 2 — Evaluate

Compute MAE, RMSE, and R² on the test set.

In [ ]:
# Your code here


### Step 3 — Log-transform the target and refit

Apply `np.log1p()` to your training and test targets, refit, and compare R² to Step 2's.

In [ ]:
# Your code here


### Step 4 — Reflect

Did the log-transform change your R²? Based on the lecture's discussion, does whichever direction it
moved automatically mean your *real-world* predictions (in hours) got better or worse? Explain.

**Your answer:**

### Big idea
> A model's R² is only ever as honest as the question you asked it, and it can't invent signal that
> isn't in your features. Before trusting any regression metric — or feeling bad about a low one —
> ask what target you're really predicting, on what scale, and whether the columns you gave the model
> ever had a real shot at explaining it.